# Quantitative Data Audit & Wrangling
This notebook profiles the raw trading data and Bitcoin Fear/Greed sentiment index, performs schema audits, applies timezone normalization, joins the datasets, evaluates wash-trading heuristics, and winsorizes extreme outliers.

In [1]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import sys

# Include src in path
sys.path.append('../')
from src.feature_engineering import parse_and_join_data, clean_and_winsorize

ROOT = Path.cwd().parent
CSV = ROOT / 'csv_files'
OUT = ROOT / 'outputs'
DATA = ROOT / 'data'
OUT.mkdir(exist_ok=True)
DATA.mkdir(exist_ok=True)

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_context('talk')
print('Environment initialized.')

Environment initialized.


## Load Datasets and Align Timezones (IST -> UTC)
We load `historical_data.csv` and `fear_greed_index.csv`. We convert `Timestamp IST` to UTC to prevent lookahead and join sentiment daily by `date_utc`.

In [2]:
merged = parse_and_join_data(ROOT / 'data' / 'historical_data.csv', ROOT / 'data' / 'fear_greed_index.csv')
print(f"Merged shape: {merged.shape}")
print(f"Null sentiment entries: {merged['fg_value'].isna().sum()}")
merged.head(3)

Merged shape: (211224, 23)
Null sentiment entries: 0


,Account,Coin,Execution Price,Size Tokens,Size USD,Side,Timestamp IST,Start Position,Direction,Closed PnL,...,Fee,Trade ID,Timestamp,timestamp_ist,timestamp_utc,date_utc,date,fg_timestamp,fg_value,fg_classification
0,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9769,986.87,7872.16,BUY,02-12-2024 22:50,0.000000,Buy,0.0,...,0.345404,8.950000e+14,1.730000e+12,2024-12-02 22:50:00,2024-12-02 17:20:00,2024-12-02,2024-12-02,1733117400,80,Extreme Greed
1,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9800,16.00,127.68,BUY,02-12-2024 22:50,986.524596,Buy,0.0,...,0.005600,4.430000e+14,1.730000e+12,2024-12-02 22:50:00,2024-12-02 17:20:00,2024-12-02,2024-12-02,1733117400,80,Extreme Greed
2,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9855,144.09,1150.63,BUY,02-12-2024 22:50,1002.518996,Buy,0.0,...,0.050431,6.600000e+14,1.730000e+12,2024-12-02 22:50:00,2024-12-02 17:20:00,2024-12-02,2024-12-02,1733117400,80,Extreme Greed


## Quantitative Data Audit & Schema Diagnostics
We check null distributions, check values bounds, and log data quality issues.

In [3]:
df_clean = clean_and_winsorize(merged)
print(f"Cleaned data shape: {df_clean.shape}")

# Generate schema audit report
schema = pd.DataFrame({
    'column': df_clean.columns,
    'dtype': [str(df_clean[c].dtype) for c in df_clean.columns],
    'null_count': [int(df_clean[c].isna().sum()) for c in df_clean.columns],
    'null_pct': [float(df_clean[c].isna().mean()) for c in df_clean.columns],
    'unique_count': [int(df_clean[c].nunique(dropna=False)) for c in df_clean.columns],
})
schema.to_csv(CSV / 'data_quality_audit.csv', index=False)
schema.to_csv(DATA / 'data_quality_audit.csv', index=False)
schema

Cleaned data shape: (211082, 34)


,column,dtype,null_count,null_pct,unique_count
0,Account,str,0,0.0,32
1,Coin,str,0,0.0,227
2,Execution Price,float64,0,0.0,60052
3,Size Tokens,float64,0,0.0,59169
4,Size USD,float64,0,0.0,118482
5,Side,str,0,0.0,2
6,Timestamp IST,str,0,0.0,27916
7,Start Position,float64,0,0.0,196788
8,Direction,str,0,0.0,11
9,Closed PnL,float64,0,0.0,90720


## Wash Trading Heuristics & Detection
We inspect accounts for self-matching trades, zero PnL trades, buy-sell symmetry, and high trading frequencies to flag wash trading behavior.

In [4]:
acc_summary = df_clean.groupby('Account').agg(
    trades=('Closed PnL', 'size'),
    total_pnl=('Closed PnL', 'sum'),
    avg_pnl=('Closed PnL', 'mean'),
    pnl_std=('Closed PnL_w', 'std'),
    volume=('Size USD', 'sum'),
    maker_rate=('Crossed', lambda x: (x == False).mean()),
    taker_rate=('Crossed', lambda x: (x == True).mean()),
    last_trade=('timestamp_utc', 'max'),
    first_trade=('timestamp_utc', 'min'),
    zero_pnl_rate=('Closed PnL', lambda x: (x == 0).mean()),
    buy_rate=('Side', lambda x: (x.astype(str).str.upper() == 'BUY').mean()),
).reset_index()

gross_gp = df_clean[df_clean['Closed PnL'] > 0].groupby('Account')['Closed PnL'].sum().rename('gross_gp')
gross_gl = df_clean[df_clean['Closed PnL'] < 0].groupby('Account')['Closed PnL'].sum().abs().rename('gross_gl')
acc_summary = acc_summary.merge(gross_gp, on='Account', how='left').merge(gross_gl, on='Account', how='left').fillna({'gross_gp':0, 'gross_gl':0})
acc_summary['profit_factor'] = acc_summary['gross_gp'] / acc_summary['gross_gl'].replace(0, np.nan)

latest = df_clean['timestamp_utc'].max()
acc_summary['recency_days'] = (latest - acc_summary['last_trade']).dt.total_seconds() / 86400
acc_summary['active_days'] = ((acc_summary['last_trade'] - acc_summary['first_trade']).dt.total_seconds() / 86400).clip(lower=1)
acc_summary['trades_per_day'] = acc_summary['trades'] / acc_summary['active_days']
acc_summary['buy_sell_symmetry'] = 1 - (acc_summary['buy_rate'] - 0.5).abs() * 2
acc_summary['near_zero_return_flag'] = ((acc_summary['total_pnl'].abs() / acc_summary['volume'].replace(0, np.nan)) < 0.0001).fillna(False).astype(int)
acc_summary['zero_pnl_flag'] = (acc_summary['zero_pnl_rate'] > 0.60).astype(int)
acc_summary['symmetry_flag'] = (acc_summary['buy_sell_symmetry'] > 0.90).astype(int)
acc_summary['high_frequency_flag'] = (acc_summary['trades_per_day'] > acc_summary['trades_per_day'].quantile(0.95)).astype(int)
acc_summary['wash_suspicion_score'] = acc_summary[['near_zero_return_flag', 'zero_pnl_flag', 'symmetry_flag', 'high_frequency_flag']].mean(axis=1)
acc_summary['human_probability'] = 1 - acc_summary['wash_suspicion_score'].clip(0, 1)

# Export wash trading report
acc_summary[['Account', 'wash_suspicion_score', 'near_zero_return_flag', 'zero_pnl_flag', 'symmetry_flag', 'high_frequency_flag', 'human_probability', 'trades', 'volume', 'total_pnl']].sort_values('wash_suspicion_score', ascending=False).to_csv(CSV / 'wash_trading_heuristics.csv', index=False)
acc_summary[['Account', 'wash_suspicion_score', 'near_zero_return_flag', 'zero_pnl_flag', 'symmetry_flag', 'high_frequency_flag', 'human_probability', 'trades', 'volume', 'total_pnl']].sort_values('wash_suspicion_score', ascending=False).to_csv(DATA / 'wash_trading_heuristics.csv', index=False)
print(f"Wash trading suspicion accounts: {(acc_summary['wash_suspicion_score'] > 0.5).sum()}")
acc_summary.sort_values('wash_suspicion_score', ascending=False).head(5)

Wash trading suspicion accounts: 0


,Account,trades,total_pnl,avg_pnl,pnl_std,volume,maker_rate,taker_rate,last_trade,first_trade,...,recency_days,active_days,trades_per_day,buy_sell_symmetry,near_zero_return_flag,zero_pnl_flag,symmetry_flag,high_frequency_flag,wash_suspicion_score,human_probability
29,0xbaaaf6571ab7d571043ff1e313a9609a10637864,21192,940163.806220,44.364091,114.126486,6.803634e+07,0.673603,0.326397,2025-04-30 22:26:00,2024-12-27 19:10:00,...,0.345139,124.136111,170.715836,0.943469,0,0,1,1,0.50,0.50
3,0x28736f43f1e871e6aa8b1148d38d4994275d72c4,13301,132464.814554,9.959012,57.784087,6.757020e+06,0.054582,0.945418,2025-05-01 03:40:00,2024-11-15 20:54:00,...,0.127083,166.281944,79.990645,0.992707,0,0,1,0,0.25,0.75
6,0x39cef799f8b69da1995852eea189df24eb5cae3c,3589,14456.919336,4.028119,168.521274,1.719338e+07,0.924770,0.075230,2025-04-14 02:32:00,2024-11-12 00:10:00,...,17.174306,153.098611,23.442407,0.918362,0,0,1,0,0.25,0.75
7,0x3f9a0aadc7f04a7c9d75dc1b5a6ddd6e36486cf6,328,53496.247243,163.098315,154.856380,1.143895e+06,0.661585,0.338415,2025-04-12 03:29:00,2024-10-02 18:31:00,...,19.134722,191.373611,1.713925,0.963415,0,0,1,0,0.25,0.75
12,0x4f93fead39b70a1824f981a54d4e55b278e9f760,7562,308975.869716,40.859015,170.377739,1.296725e+08,0.000000,1.000000,2025-05-01 06:12:00,2024-03-13 14:45:00,...,0.021528,413.643750,18.281432,0.896588,0,1,0,0,0.25,0.75


## Export Cleaned Dataset
We save the cleaned and joined table for use in downstream analysis.

In [5]:
df_clean.to_csv(CSV / 'historical_with_sentiment_clean.csv', index=False)
df_clean.to_csv(DATA / 'historical_with_sentiment_clean.csv', index=False)
print("Data saved to csv_files/historical_with_sentiment_clean.csv")

Data saved to csv_files/historical_with_sentiment_clean.csv
